# 第 1 周练习 —— Groq 技术导师助手

## 练习目标（理念）

做一个带**多轮对话记忆**的代码讲解工具：

- **输入**：先粘贴一段源码，再继续追问「这段在干什么 / 怎么改更好」
- **输出**：逐行讲解、找 bug、给改进建议（流式显示）
- **后端**：用 **Groq** 的 Chat Completions 兼容接口，模型为 `llama-3.1-8b-instant`

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user / assistant） | system 定导师风格；history 里交替存 user/assistant |
| 流式输出 `stream=True` | 逐块 `delta`，`update_display` 刷新 Markdown |
| 环境变量 | `.env` 里的 `GROQ_API_KEY` |

## 怎么跑

1. 配置 `.env`：`GROQ_API_KEY=...`
2. 运行唯一代码单元格；按提示粘贴代码，再继续提问
3. 输入 `quit` / `bye` / `exit` 结束循环


In [ ]:
# ========== 第 1 周练习：用 Groq 做代码讲解与改进建议的技术导师助手 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 GROQ_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 groq 导入 Groq 客户端：调用 Groq 的 Chat Completions 兼容 API
from groq import Groq
# 从 IPython.display 导入展示工具：流式刷新 Markdown
from IPython.display import display, Markdown, update_display

# --------------------------------------------------
# 加载环境变量
# --------------------------------------------------
# 读取 .env 到进程环境（默认不 override；保持原参数）
load_dotenv()

# 确保已设置 GROQ_API_KEY；没有就立刻失败，避免后面请求才报错
if not os.getenv("GROQ_API_KEY"):
    # 错误文案保持英文原样（可能被程序/用户依赖）
    raise ValueError("❌ GROQ_API_KEY not found in environment variables")

# 初始化 Groq 客户端：密钥从环境变量读取
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# --------------------------------------------------
# 系统提示词（system prompt）：规定「技术导师」怎么答
# --------------------------------------------------
# 发给模型的指令保持英文原样：翻译会改变回答风格/行为
system_prompt = """
You are a helpful, professional coding assistant and technical mentor.

IMPORTANT:
- Always connect new answers to topics discussed earlier in the conversation.
- Build upon previously explained concepts and reuse terminology already introduced.
- When a new concept appears, relate it to prior knowledge whenever possible.

The user will provide source code. Your responsibilities are:
1. Carefully read and fully understand the provided code.
2. Explain the code line by line using clear, beginner-friendly language.
3. Identify bugs, logical issues, or bad practices.
4. Provide corrected code when issues are found.
5. Suggest improvements for readability, modularity, and maintainability.
6. Follow industry best practices and explain WHY changes are useful.

Guidelines:
- Be precise and structured.
- Use clean formatting and code blocks.
- Prioritize learning over verbosity.
"""

# --------------------------------------------------
# 对话状态（State）
# --------------------------------------------------
# conversation_history：多轮 messages 列表（不含 system；system 每次请求时再拼）
conversation_history = []
# question_count：已提问次数；用来切换「先贴代码」还是「继续追问」的提示文案
question_count = 0

# --------------------------------------------------
# 欢迎语（给人看的 print；保持原英文文案）
# --------------------------------------------------
print("🚀 Welcome to your AI Code Assistant!")
print("📌 Paste code, ask questions, and improve step by step.")
print("❌ Type 'quit', 'bye', or 'exit' anytime to leave.\n")

# --------------------------------------------------
# 主循环：反复读入 → 调模型 → 把回答写回历史
# --------------------------------------------------
while True:

    # 第一次：请用户粘贴代码；之后：请用户提出下一个问题
    if question_count == 0:
        user_input = input("📄 Please copy-paste your code here:\n")
    else:
        user_input = input(f"\n❓ Question {question_count + 1}: What do you want to do next?\n")

    # 退出条件：quit / bye / exit（不区分大小写）
    if user_input.lower() in ["quit", "bye", "exit"]:
        print("\n👋 Thanks for using the AI Code Assistant. Happy coding!")
        break

    # 空输入：提示后 continue，不计入一次有效提问
    if not user_input.strip():
        print("⚠️ Please enter something.")
        continue

    # 把本轮用户内容追加进历史（role=user）
    conversation_history.append({"role": "user", "content": user_input})
    # 有效提问计数 +1
    question_count += 1

    # 回显一下本轮在想什么（调试/体验用）
    print(f"\n🧠 Thinking about:\n{user_input}\n")

    # --------------------------------------------------
    # 调用 Groq：流式 Chat Completions
    # --------------------------------------------------
    stream = client.chat.completions.create(
        model="llama-3.1-8b-instant",   # ✅ FREE & BEST for coding
        # system 固定在最前，后面接完整对话历史
        messages=[
            {"role": "system", "content": system_prompt}
        ] + conversation_history,
        stream=True
    )

    # 流式展示：先占位，再按 delta 刷新
    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        # 增量文本在 choices[0].delta.content
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            update_display(Markdown(response), display_id=display_handle.display_id)

    # 把助手完整回答写回历史，供下一轮「接上文」
    conversation_history.append({"role": "assistant", "content": response})
